In [ ]:
from ultralytics import YOLO
import torch
from pathlib import Path

def train_improved():
    """Увеличенное разрешение и больше эпох"""
    
    BASE_DIR = Path.cwd()
    PRETRAINED_DIR = BASE_DIR.parent / "models" / "pretrained"
    RUNS_DIR = BASE_DIR.parent / "runs" / "detect"
    DATA_CONFIG = BASE_DIR.parent / "datasets" / "data.yaml"

    model = YOLO(str(PRETRAINED_DIR / 'yolo11m.pt'))
    
    results = model.train(
        data=str(DATA_CONFIG),
        project=str(RUNS_DIR),
        name='pcb_defect_detector_v2',
        imgsz=1024,
        epochs=50,
        patience=20,  # early stopping
        batch=4,
        # Включаем Mixed Precision (смешанная точность FP16)
        # Это КРИТИЧЕСКИ важно для 3070: ускоряет обучение и экономит память.
        amp=True, 
        # Кэширование в RAM
        # Это разгрузит шину данных и ускорит обучение.
        cache=True,
        workers=8,
        # Оптимизация
        optimizer='AdamW',
        lr0=0.001,  # меньший learning rate для тонкой настройки
        lrf=0.0001,
        # Аугментации
        hsv_h=0.02,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=15,
        translate=0.15,
        scale=0.5,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        # Callback для сохранения лучших весов
        save=True,
        save_period=10,
        
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    
    return results

if __name__ == "__main__":
    results = train_improved()

In [ ]:
from ultralytics import YOLO

model = YOLO('../runs/detect/pcb_defect_detector_v2/weights/last.pt')

results = model.train(resume=True)